# 00 Setup (thin wrapper)
Run top to bottom at the start of every session. Logic lives in the repo, not here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT='/content/drive/MyDrive/selrm'
for d in ['data','cache','ckpt','results','logs','tables']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
print(os.listdir(ROOT))

In [ ]:
# Clone or update the private repo (set REPO_URL; use a token from Colab Secrets)
from google.colab import userdata
REPO_URL = 'https://github.com/<org>/<repo>.git'  # TODO
import os, subprocess
if not os.path.exists('/content/repo'):
    subprocess.run(['git','clone',REPO_URL,'/content/repo'],check=True)
else:
    subprocess.run(['git','-C','/content/repo','pull'],check=True)
%cd /content/repo

In [ ]:
# Install pinned dependencies (pin exact versions in requirements-colab.txt once verified)
!pip -q install -r requirements-colab.txt

In [ ]:
# Environment record for meta.json
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip freeze > /content/drive/MyDrive/selrm/logs/pip_freeze_$(date +%Y%m%d_%H%M).txt

In [ ]:
# Secrets -> environment (never print keys)
import os
from google.colab import userdata
for k in ['OPENROUTER_API_KEY','HF_TOKEN']:
    try:
        os.environ[k]=userdata.get(k)
    except Exception as e:
        print('missing secret', k)

In [ ]:
# Smoke test: rule library loads and labels are executed, not written
import sys; sys.path.insert(0,'/content/repo')
from selrm.rules import RULES, Mention
r=RULES[0]
print(r.rule_text())
print([r.label(s,'pen_allergy') for s in ([], [Mention('pen_allergy','finding')], [Mention('pen_allergy','finding',subject='aunt')])])  # expect [0, 1, 0]